In [ ]:
### Version 1 - SIMPLE AS OF FEB, 24TH 2026###
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import csv


# URL
url = "https://immovlan.be/en"

driver = webdriver.Safari()
wait = WebDriverWait(driver, 5)

driver.get(url)
time.sleep(3)

# COOKIES
try:
    cookie_button = wait.until(
        EC.element_to_be_clickable((By.ID, "didomi-notice-agree-button"))
    )
    cookie_button.click()
except:
    pass

# ZIP
location_input = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//input[@placeholder='Where? City, ZipCode, Province or Region']")
    )
)
zip_code = 1030
location_input.send_keys(zip_code)

# ZIP presented
first_item = wait.until(
    EC.element_to_be_clickable((By.XPATH, '//*[@id="autocomplete-chip-item-0-0"]'))
)
first_item.click()

# Pressing the SEARCH THE LIST button
search_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.search-list-button"))
)
search_button.click()

# LOADING THE FIRST PAGE WITH RESULTS
wait.until(
    EC.presence_of_element_located((By.XPATH, "//article[@itemscope and @itemtype]"))
)

# GETTING THE URL OF THE CURRENT PAGE
base_url = driver.current_url

# CHECKING IF THERE IS A PARAMETER page=
if "page=" not in base_url:
    separator = "&" if "?" in base_url else "?"
    base_url = f"{base_url}{separator}page="

print("Base URL:", base_url)

# COLLECTING LINKS OF LOTS
lots_urls = []

for page in range(1, 51):
    page_url = base_url + str(page)
    try:
        # GOING TO THE PAGE
        driver.get(page_url)
        wait.until(EC.presence_of_all_elements_located((By.XPATH, "//article[@itemscope and @itemtype]")))

        print(f"\nScraping page {page}")

        lots = driver.find_elements(By.XPATH, "//article[@itemscope and @itemtype]")

        print(f"Lots found on page {page}: {len(lots)}")
        #COLLECTING LINKS
        for lot in lots:
            url = lot.get_attribute("data-url")
            if url and url not in lots_urls:
                lots_urls.append(url)
                
        print(f"Collected so far: {len(lots_urls)}")
    except: # IF NO MORE page_url = base_url + str(page)
        print(f"No more pages with data")
        break
    
print("\nTotal lots collected:", len(lots_urls))

# PRINTING IN TERMINAL


for url in lots_urls:
    print(url)

driver.quit()

# SAVING RESULTS IN CSV
with open("../data/lots_urls.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for url in lots_urls:
        writer.writerow([url])

print(f"Total lots collected: {len(lots_urls)}")
print(f"n\Data saved in 'lots_urls.csv")





In [ ]:
### Version 2 - SIMPLIFIED ###
# USING SELENIUM TO GO THRU THE COOKIES, SELECT THE PROPERTY LOT TYPE (House, Apartment) AND FIRST PROVINCE TO MAKE A PATTERN
# USE REGULAR MANIPULATION TO WORK WITH URL LINK AND GET THE DATA FOR ALL PROVINCES 

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import csv


# URL
url = "https://immovlan.be/en"


# IMPORTANT: You need to choose your correct browser to run.  I did it with Safari browser
driver = webdriver.Safari()
wait = WebDriverWait(driver, 5)

driver.get(url)
time.sleep(3)

# COOKIES
try:
    cookie_button = wait.until(
        EC.element_to_be_clickable((By.ID, "didomi-notice-agree-button"))
    )
    cookie_button.click()
except:
    pass

# Choosing 'House' and 'Apartment' to narrow the search
driver.find_element(By.XPATH, "//input[@placeholder='What?']").click()
time.sleep(3)
driver.find_element(By.XPATH, "//span[contains(text(), 'House')]").click()
driver.find_element(By.XPATH, "//span[contains(text(), 'Apartment')]").click()
time.sleep(1)

# Creating the list of provinces of Belgium from the site
ul = driver.find_element(By.ID, "button-group-spotlight-province")
links = ul.find_elements(By.TAG_NAME, "a")
provinces_list = [link.text.strip() for link in links]

print("Provinces list:", provinces_list)

province_to_select = provinces_list[0]  # Select the first province aka Brussels for testing

# Applying the search pattern for the province selection
# Finding the input field for the location
location_input = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//input[@placeholder='Where? City, ZipCode, Province or Region']")
    )
)
# Slowly typing the province name to trigger the dropdown suggestions    
for ch in province_to_select: 
    location_input.send_keys(ch)
    time.sleep(0.15)
# Waiting for the dropdown suggestions to appear and selecting the correct province
    items = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li[role='option']"))
    )
    for item in items:
        value_type = item.get_attribute("data-value-type")
        text = item.text.strip()
        if value_type == "province" and province_to_select.lower() in text.lower():
            item.click()
            break
    

# Pressing the 'SEARCH THE LIST' button
search_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.search-list-button"))
)
search_button.click()

# LOADING THE FIRST PAGE WITH RESULTS
wait.until(
    EC.presence_of_element_located((By.XPATH, "//article[@itemscope and @itemtype]"))
)

# GETTING THE URL OF THE CURRENT PAGE
base_province_url = driver.current_url
print(f"Base url before changing pages and provinces: {base_province_url}")

# INITIALIZING THE LIST OF LOTS URLS
lots_urls = []

# THE LOOP TO SCRAPE LOTS LINKS FOR EACH PROVINCE
# We will replace the province parameter in the URL to navigate through different provinces without having to interact with the page elements again
for province in provinces_list:
    province_url = base_province_url.replace("provinces=brussels", f"provinces={province.lower()}")
    print(f"New URL for {province}: {province_url}")
    

    # CHECKING IF THERE IS A PARAMETER page=
    if "page=" not in province_url:
        separator = "&" if "?" in province_url else "?"
        base_url = f"{province_url}{separator}page="

    print("Base URL:", base_url)

    # COLLECTING LINKS OF LOTS FOR THE CURRENT PROVINCE, MAX PAGE = 50 - NO MORE ALLOWED BY THE SITE
    for page in range(1, 51):
        page_url = base_url + str(page)
        try:
            # GOING TO THE PAGE
            driver.get(page_url)
            wait.until(EC.presence_of_all_elements_located((By.XPATH, "//article[@itemscope and @itemtype]")))

            print(f"\nScraping page {page}")

            # A LIST OF ALL LOTS ON THE PAGE
            lots = driver.find_elements(By.XPATH, "//article[@itemscope and @itemtype]")

            print(f"Lots found on page {page}: {len(lots)}")

            #COLLECTING LINKS
            for lot in lots:
                url = lot.get_attribute("data-url")
                if url and url not in lots_urls:
                    lots_urls.append(url)
                    
            print(f"Collected so far: {len(lots_urls)}")
        except: # IF NO MORE page_url = base_url + str(page)
            print(f"No more pages with data")
            break
    
print("\nTotal lots collected:", len(lots_urls))

# PRINTING IN TERMINAL TO CHECK
for url in lots_urls:
    print(url)

driver.quit()

# SAVING RESULTS IN CSV
with open("../data/lots_urls.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for url in lots_urls:
        writer.writerow([url])

print(f"Total lots collected: {len(lots_urls)}")
print(f"Data saved in 'lots_urls.csv")



In [ ]:
checking_nonduplicates = {l for l in lots_urls}
print(f"Total nonduplicates found: {len(checking_nonduplicates)}")
new_set = set(lots_urls)
print(f"Total unique lots collected: {len(new_set)}")

In [ ]:
import csv
with open("../data/lots_urls.csv", "r", newline="") as f:
    reader = csv.reader(f)
    csv_lots_urls = [row[0] for row in reader]
    my_set = set(csv_lots_urls)

with open("../data/all_properties_urls_without_projects.csv", "r", newline="") as f:
    reader = csv.reader(f)
    csv_lots_urls = [row[0] for row in reader]
    Simon_set = set(csv_lots_urls)

only_in_mine = my_set - Simon_set
only_in_Simon = Simon_set - my_set
print(f"Total lots only in my set: {len(only_in_mine)}")
print(f"Total lots only in Simon's set: {len(only_in_Simon)}")



In [ ]:
for url in only_in_mine:
    print(url)

In [ ]:
for url in only_in_Simon:
    print(url)
    

In [117]:
### TESTING SELENIUM @ FEB 26TH, 2026 ###

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import csv
import unicodedata

# URL
url = "https://immovlan.be/en"

driver = webdriver.Safari()
wait = WebDriverWait(driver, 10)

# RUNNING DRIVER
driver.get(url)
time.sleep(3)

# COOKIES
try:
    cookie_button = wait.until(
        EC.element_to_be_clickable((By.ID, "didomi-notice-agree-button"))
    )
    cookie_button.click()
except:
    pass
time.sleep(3)

# # MAXIMALIZING THE WINDOW TO AVOID ISSUES WITH ELEMENTS NOT BEING CLICKABLE
# driver.fullscreen_window()
# time.sleep(1)
# driver.maximize_window()
# time.sleep(5)

# FUNCTION TO SELECT THE PROPERTY TYPE (House, Apartment) TO NARROW THE SEARCH
def select_property_type(driver):
    driver.find_element(By.XPATH, "//input[@placeholder='What?']").click()
    time.sleep(3)
    driver.find_element(By.XPATH, "//span[contains(text(), 'House')]").click()
    driver.find_element(By.XPATH, "//span[contains(text(), 'Apartment')]").click()
    time.sleep(1)

# RUNNING IT TO GET THE ERASE OPTIONS BUTTON TO APPEAR AND AVOID ISSUES WITH THE FILTERS
select_property_type(driver)

# CREATING LIST OF AREAS TO SEARCH = PROVINCES + MUNICIPALITIES OF BRUSSELS
ul = driver.find_element(By.ID, "button-group-spotlight-province")
links = ul.find_elements(By.TAG_NAME, "a")
provinces_list = [link.text.strip() for link in links]

# chosen = ['Brussels']

bx_municipalities=["1000-brussels","1020-laken","1030-schaarbeek","1040-etterbeek","1050-elsene","1060-sint-gillis","1070-anderlecht","1080-sint-jans-molenbeek","1090-jette","1120-neder-over-heembeek","1130-haren","1140-evere","1150-sint-pieters-woluwe","1160-oudergem","1170-watermaal-bosvoorde","1180-ukkel","1190-vorst","1200-sint-lambrechts-woluwe","1210-sint-joost-ten-node"]

area_list = provinces_list[1:] + bx_municipalities
print(f"Area list to search: {area_list}")


# FUNCTION TO NORMALIZE THE PROVINCE NAMES TO MATCH THE SLUGS IN THE HTML ATTRIBUTES
def normalize_slug(text):
    # "Liège" → "Liege"
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    return text.lower().replace(" ", "-")


# FUNCTION TO REMOVE THE APPLIED FILTER
def remove_selected_filter(driver): 
    driver.find_element(By.XPATH, "//a[contains(text(), 'Erase options')]").click()
time.sleep(1)

lots_urls = []

# FUNCTION TO GET LOTS LINKS FOR THE PAGE
def get_lots_links_for_page(page_url, lots_urls):
    try:
        # GOING TO THE PAGE
        driver.get(page_url)
        wait.until(EC.presence_of_all_elements_located((By.XPATH, "//article[@itemscope and @itemtype]")))

        # A LIST OF ALL LOTS ON THE PAGE
        lots = driver.find_elements(By.XPATH, "//article[@itemscope and @itemtype]")

        #COLLECTING LINKS
        for lot in lots:
            url = lot.get_attribute("data-url")
            if url and url not in lots_urls:
                lots_urls.append(url)
                
        print(f"Collected so far: {len(lots_urls)}")
    except:
        print(f"No more pages with data")
    
    return lots_urls

# FUNCTION TO SELECT THE AREA (PROVINCE OR MUNICIPALITY) TO NARROW THE SEARCH
def select_area(driver, area):
    if area[:4].isdigit():
        mode = 'city'
        area_fixed = area[:4]  # For Brussels municipalities, we take only the first 4 characters to match the city code
    else:
        mode = 'province'
        area_fixed = area

    # CLEARING THE INPUT FIELD
    input_box = wait.until(
        EC.presence_of_element_located((By.XPATH, "//input[@placeholder='Where? City, ZipCode, Province or Region']"))
    )
    input_box.clear()
    time.sleep(0.3)

    # INPUTTING THE AREA NAME
    input_box.send_keys(area_fixed)
    time.sleep(0.5)

    # WAITING FOR THE DROPDOWN SUGGESTIONS TO APPEAR
    wait.until(EC.presence_of_element_located((By.XPATH, "//li[@role='option']")))

    # CREATING A SLUG TO FIND THE CORRECT ELEMENT IN THE DROPDOWN
    slug = normalize_slug(area)

    xpath = f"//li[@role='option' and @data-value-type='{mode}' and @data-value-slug='{slug}']"

    # WAITING FOR THE CORRECT OPTION TO BE PRESENT IN THE DROPDOWN
    option = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))

    # CLICKING WITH JS (Safari fix)
    driver.execute_script("arguments[0].click();", option)

# FINALLY...
for area in area_list:

    # REMOVING THE PREVIOUSLY APPLIED FILTER
    try:
        remove_selected_filter(driver)
    except:
        pass

    # SELECTING THE PROPERTY TYPE AGAIN TO AVOID ISSUES WITH THE FILTERS
    select_property_type(driver)

    # SETTING THE AREA
    select_area(driver, area)
    
    time.sleep(2)

    # PRESSING THE 'SEARCH THE LIST' BUTTON
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.search-list-button"))).click()

    # LOADING THE FIRST PAGE WITH RESULTS
    wait.until(EC.presence_of_element_located((By.XPATH, "//article[@itemscope and @itemtype]")))

    # GETTING THE URL OF THE CURRENT PAGE
    base_area_url = driver.current_url
    print(f"Base url for the {area}: {base_area_url}")

    # LOOPING THRU THE PAGES FOR THE CURRENT AREA
    for page in range(1, 3):
        page_url = base_area_url.replace("noindex=1", f"page={page}&noindex=1")
        
        print(f"Page with lots: {page_url}")
        
        lots_urls = get_lots_links_for_page(page_url, lots_urls)


    # CLICKING ON THE LOGO TO GO BACK TO THE HOMEPAGE AND START A NEW SEARCH
    driver.find_element(By.XPATH, "//a[@href='/en']/img[@id='logo']").click()
    time.sleep(5)

    # CONTINUE MY PREVIOUS SEARCH BUTTON CLOSE - CRAZY POP-UP THAT APPEARS RANDOMLY AND BLOCKS THE INTERACTION WITH THE PAGE
    try:
        close_btn = driver.find_element(By.XPATH, "//div[@id='welcome-back-modal']//div[@class='modal-header']//button[contains(@class,'btn-close')]")
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(5)
    except:
        pass

driver.quit()

# SAVING RESULTS IN CSV
with open("../data/lots_urls.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for url in lots_urls:
        writer.writerow([url])

print(f"Total lots collected: {len(lots_urls)}")
print(f"Data saved in 'data/lots_urls.csv")

print(lots_urls)

Python(11296) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(11329) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Area list to search: ['Hainaut', 'Liège', 'East Flanders', 'West Flanders', 'Antwerp', 'Brabant Wallon', 'Vlaams-Brabant', 'Namur', 'Limburg', 'Luxembourg', '1000-brussels', '1020-laken', '1030-schaarbeek', '1040-etterbeek', '1050-elsene', '1060-sint-gillis', '1070-anderlecht', '1080-sint-jans-molenbeek', '1090-jette', '1120-neder-over-heembeek', '1130-haren', '1140-evere', '1150-sint-pieters-woluwe', '1160-oudergem', '1170-watermaal-bosvoorde', '1180-ukkel', '1190-vorst', '1200-sint-lambrechts-woluwe', '1210-sint-joost-ten-node']
Base url for the Hainaut: https://immovlan.be/en/real-estate?transactiontypes=for-sale,in-public-sale&propertytypes=house,apartment&provinces=hainaut&noindex=1
Page with lots: https://immovlan.be/en/real-estate?transactiontypes=for-sale,in-public-sale&propertytypes=house,apartment&provinces=hainaut&page=1&noindex=1
Collected so far: 20
Page with lots: https://immovlan.be/en/real-estate?transactiontypes=for-sale,in-public-sale&propertytypes=house,apartment&pro

In [ ]:
# def get_lots_links_for_page(page_url, lots_urls):
#     try:
#         # GOING TO THE PAGE
#         driver.get(page_url)
#         wait.until(EC.presence_of_all_elements_located((By.XPATH, "//article[@itemscope and @itemtype]")))

#         # A LIST OF ALL LOTS ON THE PAGE
#         lots = driver.find_elements(By.XPATH, "//article[@itemscope and @itemtype]")

#         #COLLECTING LINKS
#         for lot in lots:
#             url = lot.get_attribute("data-url")
#             if url and url not in lots_urls:
#                 lots_urls.append(url)
                
#         print(f"Collected so far: {len(lots_urls)}")
#     except: # IF NO MORE page_url = base_url + str(page)
#         print(f"No more pages with data")
    
#     return lots_urls



In [116]:
# APPLYING THE BS4 TO GET THE LOTS LINKS FOR THE PAGE
from requests import Session
from bs4 import BeautifulSoup
session=Session()
headers={"User-Agent":"Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Mobile Safari/537.36"}

page_urls=['https://immovlan.be/en/real-estate?transactiontypes=for-sale,in-public-sale&propertytypes=house,apartment&provinces=antwerp&page=1&noindex=1', 'https://immovlan.be/en/real-estate?transactiontypes=for-sale,in-public-sale&propertytypes=house,apartment&provinces=west-flanders&page=1&noindex=1']


property_urls=[]
for page_url in page_urls:
    r=session.get(page_url,headers=headers)
    html=r.text
    soup=BeautifulSoup(html,"html.parser")
    for h2 in soup.find_all("h2"):
        for a in h2.find_all("a",href=True):
            property_urls.append(a["href"])

for url in property_urls:
    print(url)


https://immovlan.be/en/projectdetail/1485769-00955250_om_293210
https://immovlan.be/en/projectdetail/1485768-01267924_om_292931
https://immovlan.be/en/projectdetail/1485768-01267924_om_290927
https://immovlan.be/en/detail/residence/for-sale/2627/schelle/rbv31460
https://immovlan.be/en/detail/apartment/for-sale/2440/geel/rbv31396
https://immovlan.be/en/detail/apartment/for-sale/2250/olen/rbv30913
https://immovlan.be/en/detail/apartment/for-sale/2300/turnhout/rbv30696
https://immovlan.be/en/detail/apartment/for-sale/2300/turnhout/rbv30659
https://immovlan.be/en/detail/residence/for-sale/2500/lier/rbv29770
https://immovlan.be/en/detail/residence/for-sale/2260/westerlo/rbv29174
https://immovlan.be/en/detail/villa/for-sale/2340/beerse/rbv28950
https://immovlan.be/en/projectdetail/1485769-00955250_om_295379
https://immovlan.be/en/projectdetail/1485771-01267921_om_295361
https://immovlan.be/en/projectdetail/1485771-01267921_om_295684
https://immovlan.be/en/projectdetail/1737550-01335585_om_29

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Safari()
wait = WebDriverWait(driver, 0.5)

driver.get("https://www.selenium.dev/selenium/web/inputs.html")

is_email_visible = driver.find_element(By.NAME, 'email_input').is_displayed()
print(f"Is the email input visible? {is_email_visible}")

is_enabled_button = driver.find_element(By.NAME, 'button_input').is_enabled()
print(f"Is the button enabled? {is_enabled_button}")

is_selected_checkbox = driver.find_element(By.NAME, 'checkbox_input').is_selected()
print(f"Is the checkbox selected? {is_selected_checkbox}")

tag_name_inp = driver.find_element(By.NAME, 'email_input').tag_name
print(f"Tag name of the email input: {tag_name_inp}")

rect = driver.find_element(By.NAME, 'range_input').rect
print(f"Rectangle of the range input: {rect}")

css_value = driver.find_element(By.NAME, 'color_input').value_of_css_property('font-size')
print(f"Font size of the color input: {css_value}")

text = driver.find_element(By.TAG_NAME, 'h1').text
print(f"Text of the h1 element: {text}")

email_txt = driver.find_element(By.NAME, 'email_input')
value_info = email_txt.get_attribute('value')
print(f"Value of the email input: {value_info}")

check_input = driver.find_element(By.NAME, 'checkbox_input')
check_input.click()

email_input = driver.find_element(By.NAME, 'email_input')
email_input.clear()
email = 'Vasya Pupkin'
email_input.send_keys(email)


driver.quit()

Is the email input visible? True
Is the button enabled? True
Is the checkbox selected? True
Tag name of the email input: INPUT
Rectangle of the range input: {'y': 124.4375, 'x': 10, 'width': 129, 'height': 17}
Font size of the color input: 11px
Text of the h1 element: Testing Inputs
Value of the email input: admin@localhost


In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common import NoSuchElementException, ElementNotInteractableException
import time


driver2 = webdriver.Safari()


driver2.get("https://www.selenium.dev/selenium/web/dynamic.html")

revealed = driver2.find_element(By.ID, 'revealed')
print(f"Is the revealed element visible? {revealed.is_displayed()}")

driver2.find_element(By.ID, 'reveal').click()

errors = [NoSuchElementException, ElementNotInteractableException]
wait2 = WebDriverWait(driver2, timeout=2, poll_frequency=0.2, ignored_exceptions=errors)
wait2.until(lambda _: revealed.send_keys("I am revealed!") or True)



title = driver2.title
print(f"Title of the page: {title}")

url = driver2.current_url
print(f"Current URL: {url}")






driver2.quit()

Is the revealed element visible? False
Title of the page: 
Current URL: https://www.selenium.dev/selenium/web/dynamic.html


In [18]:
# SELENIUM ALERTS

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

url = "https://www.selenium.dev/documentation/webdriver/interactions/alerts/"

driver3 = webdriver.Safari()
driver3.get(url)


element = driver.find_element(By.LINK_TEXT, "See a sample confirm")
driver.execute_script("arguments[0].click();", element)

wait = WebDriverWait(driver, timeout=2)
alert = wait.until(lambda d : d.switch_to.alert)
text = alert.text
alert.dismiss()



# def test_prompt_popup():
#     driver = webdriver.Chrome()
#     driver.get(url)
#     element = driver.find_element(By.LINK_TEXT, "See a sample prompt")
#     driver.execute_script("arguments[0].click();", element)

#     wait = WebDriverWait(driver, timeout=2)
#     alert = wait.until(lambda d : d.switch_to.alert)
#     alert.send_keys("Selenium")
#     text = alert.text
#     alert.accept()
#     assert text == "What is your tool of choice?"
    
#     driver.quit()

InvalidSessionIdException: Message: ; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception


In [38]:
import sys

from selenium.webdriver import Keys, ActionChains
from time import sleep
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.actions.mouse_button import MouseButton
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver4 = webdriver.Safari()
driver4.get("https://selenium.dev/selenium/web/mouse_interaction.html")
cmd_ctrl = Keys.COMMAND if sys.platform == 'darwin' else Keys.CONTROL

# clickable = driver4.find_element(By.ID, "clickable")
# ActionChains(driver4) \
#     .click_and_hold(clickable) \
#     .perform()

# sleep(0.5)

# clickable2 = driver4.find_element(By.ID, "click")
# ActionChains(driver4) \
#     .click(clickable2) \
#     .perform()

mouse_tracker = driver4.find_element(By.ID, "mouse-tracker")
ActionChains(driver4) \
    .move_to_element_with_offset(mouse_tracker, 8, 0) \
    .perform()



# draggable = driver4.find_element(By.ID, "draggable")
# droppable = driver4.find_element(By.ID, "droppable")
# ActionChains(driver4) \
#     .drag_and_drop(draggable, droppable) \
#     .perform()

# draggable = driver4.find_element(By.ID, "draggable")
# start = draggable.location
# finish = driver4.find_element(By.ID, "droppable").location
# ActionChains(driver4) \
#     .drag_and_drop_by_offset(draggable, finish['x'] - start['x'], finish['y'] - start['y']) \
#     .perform()


In [39]:
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait

driver5 = webdriver.Safari()
driver5.get("https://www.selenium.dev/selenium/web/dynamic.html")

revealed = driver5.find_element(By.ID, "revealed")
driver5.find_element(By.ID, "reveal").click()

wait = WebDriverWait(driver5, timeout=2)
wait.until(EC.visibility_of_element_located((By.ID, "revealed")))

revealed.send_keys("Displayed")

In [109]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Safari()
driver.get("https://immovlan.be/en")
wait = WebDriverWait(driver, 20)

# Cookies
try:
    cookie_button = wait.until(
        EC.element_to_be_clickable((By.ID, "didomi-notice-agree-button"))
    )
    cookie_button.click()
except:
    pass

driver.set_window_size(1400, 900)
wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

# 1. Search
search_menu = wait.until(
    EC.presence_of_element_located((
        By.XPATH,
        "//li[@class='dropdown_1st']/span[@class='dropbtn']"
    ))
)
driver.execute_script("arguments[0].click();", search_menu)

# 2. In Belgium
in_be = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//a[normalize-space()='In Belgium']"))
)
driver.execute_script("arguments[0].click();", in_be)

# 3. Wait for form
wait.until(EC.presence_of_element_located((By.ID, "sale-form")))

# 4. Search the list
# search_btn = wait.until(
#     EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'search-list-button')]"))
# )
s_btn = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//button[@class='button button-primary button-block waves-effect waves-light search-list-button mr-md-1 mr-0']"))
)
driver.execute_script("arguments[0].click();", s_btn)

wait.until(EC.presence_of_element_located((By.ID, "search-results")))
print("OK")
driver.quit()



TimeoutException: Message: 


In [45]:
from selenium.webdriver import Keys, ActionChains
from time import sleep
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.actions.mouse_button import MouseButton
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.select import Select

# driver6 = webdriver.Safari()
# driver6.get("https://selenium.dev/selenium/web/formPage.html")

# select_element = driver6.find_element(By.NAME, 'selectomatic')
# select = Select(select_element)

# two_element = driver6.find_element(By.CSS_SELECTOR, 'option[value=two]')
# four_element = driver6.find_element(By.CSS_SELECTOR, 'option[value=four]')
# count_element = driver6.find_element(By.CSS_SELECTOR, "option[value='still learning how to count, apparently']")

# select.select_by_visible_text('Four')
# time.sleep(1)
# select.select_by_value('two')
# time.sleep(1)

# select.select_by_index(3)
# time.sleep(1)


driver7 = webdriver.Safari()
driver7.get("https://selenium.dev/selenium/web/formPage.html")

select_element = driver7.find_element(By.NAME, 'multi')
select = Select(select_element)

ham_element = driver7.find_element(By.CSS_SELECTOR, 'option[value=ham]')
print(f"Ham element: {ham_element.text}")
gravy_element = driver7.find_element(By.CSS_SELECTOR, "option[value='onion gravy']")
print(f"Gravy element: {gravy_element.text}")
egg_element = driver7.find_element(By.CSS_SELECTOR, 'option[value=eggs]')
print(f"Egg element: {egg_element.text}")
sausage_element = driver7.find_element(By.CSS_SELECTOR, "option[value='sausages']")
print(f"Sausage element: {sausage_element.text}")

time.sleep(1)

option_elements = select_element.find_elements(By.TAG_NAME, 'option')
print(f"Option elements: {[opt.text for opt in option_elements]}")

option_list = select.options
print(f"Option list from Select class: {[opt.text for opt in option_list]}")


selected_option_list = select.all_selected_options
print(f"Selected options from all_selected_options? {[opt.text for opt in selected_option_list]}")
expected_selection = [egg_element, sausage_element]


select.select_by_value('ham')
time.sleep(2)
select.select_by_value('onion gravy')
time.sleep(2)

select.deselect_by_value('eggs')
time.sleep(2)
select.deselect_by_value('sausages')
time.sleep(2)


Ham element: Ham
Gravy element: Onion gravy
Egg element: Eggs
Sausage element: Sausages
Option elements: ['Eggs', 'Ham', 'Sausages', 'Onion gravy']
Option list from Select class: ['Eggs', 'Ham', 'Sausages', 'Onion gravy']
Selected options from all_selected_options? ['Eggs', 'Sausages']


### TESTING SELENIUM VER. 2.0 @ FEB 28TH, 2026 ###

In [ ]:
from selenium.webdriver import Keys, ActionChains
from time import sleep
from selenium.webdriver.common.actions.action_builder import ActionBuilder
from selenium.webdriver.common.actions.mouse_button import MouseButton
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.select import Select

driver = webdriver.Safari()

driver.get("https://immovlan.be/en")

wait = WebDriverWait(driver, 10)

driver.get(url)

# COOKIES
try:
    cookie_button = wait.until(EC.element_to_be_clickable((By.ID, "didomi-notice-agree-button")))
    cookie_button.click()
except:
    pass
time.sleep(3)

# # MAXIMALIZING THE WINDOW TO AVOID ISSUES WITH ELEMENTS NOT BEING CLICKABLE
# driver.fullscreen_window()
# time.sleep(1)
# driver.maximize_window()
# time.sleep(5)

# FUNCTION TO SELECT THE PROPERTY TYPE (House, Apartment) TO NARROW THE SEARCH
def select_property_type(driver):
    driver.find_element(By.XPATH, "//input[@placeholder='What?']").click()
    time.sleep(3)
    driver.find_element(By.XPATH, "//span[contains(text(), 'House')]").click()
    driver.find_element(By.XPATH, "//span[contains(text(), 'Apartment')]").click()
    time.sleep(1)

# RUNNING IT TO GET THE ERASE OPTIONS BUTTON TO APPEAR AND AVOID ISSUES WITH THE FILTERS
select_property_type(driver)

# CREATING LIST OF AREAS TO SEARCH = PROVINCES + MUNICIPALITIES OF BRUSSELS
ul = driver.find_element(By.ID, "button-group-spotlight-province")
links = ul.find_elements(By.TAG_NAME, "a")
provinces_list = [link.text.strip() for link in links]

# chosen = ['Brussels']

bx_municipalities=["1000-brussels","1020-laken","1030-schaarbeek","1040-etterbeek","1050-elsene","1060-sint-gillis","1070-anderlecht","1080-sint-jans-molenbeek","1090-jette","1120-neder-over-heembeek","1130-haren","1140-evere","1150-sint-pieters-woluwe","1160-oudergem","1170-watermaal-bosvoorde","1180-ukkel","1190-vorst","1200-sint-lambrechts-woluwe","1210-sint-joost-ten-node"]

area_list = provinces_list[1:] + bx_municipalities
print(f"Area list to search: {area_list}")


# FUNCTION TO NORMALIZE THE PROVINCE NAMES TO MATCH THE SLUGS IN THE HTML ATTRIBUTES
def normalize_slug(text):
    # "Liège" → "Liege"
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    return text.lower().replace(" ", "-")


# FUNCTION TO REMOVE THE APPLIED FILTER
def remove_selected_filter(driver): 
    driver.find_element(By.XPATH, "//a[contains(text(), 'Erase options')]").click()
time.sleep(1)

lots_urls = []

# FUNCTION TO GET LOTS LINKS FOR THE PAGE
def get_lots_links_for_page(page_url, lots_urls):
    try:
        # GOING TO THE PAGE
        driver.get(page_url)
        wait.until(EC.presence_of_all_elements_located((By.XPATH, "//article[@itemscope and @itemtype]")))

        # A LIST OF ALL LOTS ON THE PAGE
        lots = driver.find_elements(By.XPATH, "//article[@itemscope and @itemtype]")

        #COLLECTING LINKS
        for lot in lots:
            url = lot.get_attribute("data-url")
            if url and url not in lots_urls:
                lots_urls.append(url)
                
        print(f"Collected so far: {len(lots_urls)}")
    except:
        print(f"No more pages with data")
    
    return lots_urls

# FUNCTION TO SELECT THE AREA (PROVINCE OR MUNICIPALITY) TO NARROW THE SEARCH
def select_area(driver, area):
    if area[:4].isdigit():
        mode = 'city'
        area_fixed = area[:4]  # For Brussels municipalities, we take only the first 4 characters to match the city code
    else:
        mode = 'province'
        area_fixed = area

    # CLEARING THE INPUT FIELD
    input_box = wait.until(
        EC.presence_of_element_located((By.XPATH, "//input[@placeholder='Where? City, ZipCode, Province or Region']"))
    )
    input_box.clear()
    time.sleep(0.3)

    # INPUTTING THE AREA NAME
    input_box.send_keys(area_fixed)
    time.sleep(0.5)

    # WAITING FOR THE DROPDOWN SUGGESTIONS TO APPEAR
    wait.until(EC.presence_of_element_located((By.XPATH, "//li[@role='option']")))

    # CREATING A SLUG TO FIND THE CORRECT ELEMENT IN THE DROPDOWN
    slug = normalize_slug(area)

    xpath = f"//li[@role='option' and @data-value-type='{mode}' and @data-value-slug='{slug}']"

    # WAITING FOR THE CORRECT OPTION TO BE PRESENT IN THE DROPDOWN
    option = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))

    # CLICKING WITH JS (Safari fix)
    driver.execute_script("arguments[0].click();", option)

# FINALLY...
for area in area_list:

    # REMOVING THE PREVIOUSLY APPLIED FILTER
    try:
        remove_selected_filter(driver)
    except:
        pass

    # SELECTING THE PROPERTY TYPE AGAIN TO AVOID ISSUES WITH THE FILTERS
    select_property_type(driver)

    # SETTING THE AREA
    select_area(driver, area)
    
    time.sleep(2)

    # PRESSING THE 'SEARCH THE LIST' BUTTON
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.search-list-button"))).click()

    # LOADING THE FIRST PAGE WITH RESULTS
    wait.until(EC.presence_of_element_located((By.XPATH, "//article[@itemscope and @itemtype]")))

    # GETTING THE URL OF THE CURRENT PAGE
    base_area_url = driver.current_url
    print(f"Base url for the {area}: {base_area_url}")

    # LOOPING THRU THE PAGES FOR THE CURRENT AREA
    for page in range(1, 3):
        page_url = base_area_url.replace("noindex=1", f"page={page}&noindex=1")
        
        print(f"Page with lots: {page_url}")
        
        lots_urls = get_lots_links_for_page(page_url, lots_urls)


    # CLICKING ON THE LOGO TO GO BACK TO THE HOMEPAGE AND START A NEW SEARCH
    driver.find_element(By.XPATH, "//a[@href='/en']/img[@id='logo']").click()
    time.sleep(5)

    # CONTINUE MY PREVIOUS SEARCH BUTTON CLOSE - CRAZY POP-UP THAT APPEARS RANDOMLY AND BLOCKS THE INTERACTION WITH THE PAGE
    try:
        close_btn = driver.find_element(By.XPATH, "//div[@id='welcome-back-modal']//div[@class='modal-header']//button[contains(@class,'btn-close')]")
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(5)
    except:
        pass

driver.quit()

# SAVING RESULTS IN CSV
with open("../data/lots_urls.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for url in lots_urls:
        writer.writerow([url])

print(f"Total lots collected: {len(lots_urls)}")
print(f"Data saved in 'data/lots_urls.csv")

print(lots_urls)

